In [ ]:
# Import libararies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load dataset
df = pd.read_csv("active_subscriber_raw.csv")
print(df.info())
print(df.shape)
df.head()

Columns that are either empty (opted_in_at, optin_ip, unsubscribed_at), irrelevant to bot detection (fields, groups as raw JSON), or redundant (created_at, updated_at vs subscribed_at) will be dropped. `source` column is kept as it can be used for analyzing the sources bot-behvaior-subscribers are coming from. 

In [ ]:
# Dropping irrevelant columns
# source column is kept, can be used for analyzing bot behavior subscribers coming from different sources
dropped_df = df.drop(columns = [
    'status',
    'unsubscribed_at', 
    'opted_in_at', 
    'optin_ip', 
    'ip_address', 
    'created_at', 
    'updated_at',
    'fields',
    'groups'
    ]).drop_duplicates(subset=['email'])
dropped_df.info()

In [ ]:
# Converting subscribed_at to datetime
dropped_df['subscribed_at'] = pd.to_datetime(df['subscribed_at'])
dropped_df.info()
dropped_df.head()

Subscribers who have never received a campaign are excluded. There is no engagement history to analyze, and they contribute no signal to bot detection.

In [ ]:
# Filtering out the new subscribers who hasn't recieved any email
emails_sent_df = dropped_df[dropped_df['sent'] > 0]
print(emails_sent_df.shape)
print(emails_sent_df.info())
emails_sent_df.head()

School and institution security gateways automatically scan emails and click links for threats before a human ever sees the message, artificially inflating both open counts and click counts. One key metric for measuring email marketing effectiveness is click-to-open rate (CTOR), clicks divided by opens. CTOR is more meaningful than click-through rate / click rate (CTR) because it measures the engagement of emails opened, rather than raw click volume relative to total sends. Due to the metric inflation, our reported CTOR is corrupted. The goal of this pipeline is to identify and remove bot-contaminated signals so we can compare our current inflated CTOR against a cleaned baseline reflecting actual human engagement.

In [ ]:
# Compute CTOR
emails_sent_df['ctor'] = emails_sent_df['clicks_count'] / emails_sent_df['opens_count'] * 100
print(emails_sent_df.info())
emails_sent_df.head()


The rows where the CTOR are NaNs have `opens_count` of 0, which is undefined. These contacts doesn't provide much insights to engagement signals and are excluded.

In [ ]:
# Dropping the NaNs in ctor column
emails_sent_df = emails_sent_df[emails_sent_df['ctor'].notna()]
emails_sent_df.info()
emails_sent_df.head()

In [ ]:
emails_sent_df.describe()

The median subscriber has opened emails but never clicked. More than half the active list has no click engagement. The upper quarter of the distribution, high `clicks_count` and `ctor`, represents the primary targets for bot detection. The maximum values for `open_rate`, `click_rate`, and `ctor` are consistent with bot scanning behavior.

In [ ]:
# Reshaping the data from wide to long format using melt
# Allows for graphing multiple columns in boxplot
counts_melt = emails_sent_df[['sent', 'opens_count', 'clicks_count']].melt()
rate_melt = emails_sent_df[['open_rate', 'click_rate', 'ctor']].melt()
print(counts_melt.info())
print(rate_melt.info())

Two side-by-side boxplots show the spread of count-based metrics (`sent`, `opens_count`, `clicks_count`) and rate-based metrics (`open_rate`, `click_rate`, `ctor`). Outliers are hidden with `showfliers=False` because the extreme high-CTOR contacts collapse the visible range when included, making the IQR unreadable. The outliers are not removed from the data.

In [ ]:
# Creating boxplot
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(20, 8)) # creating subplot for two plots
sns.boxplot(data=counts_melt, x='variable', y='value', ax=axs[0], showfliers=False) # showfliers=False to hide big outliers to make the graph more readable
axs[0].set_xlabel('Sent, Open and Click')
axs[0].set_ylabel('Counts')

sns.boxplot(data=rate_melt, x='variable', y='value', ax=axs[1], showfliers=False) # showfliers=False
axs[1].set_xlabel('Open, Click, CTOR')
axs[1].set_ylabel('Rate')


The CTOR distribution is plotted on a log scale on the y-axis. The raw count scale is unusable here due to the high cluster at zero is drastically larger than the high-CTOR tail, which collapses it to invisible. Log scaling makes both the low-engagement majority and the high-CTOR outlier population visible in the same plot.

In [ ]:
# Plotting the histogram
plt.hist(emails_sent_df['ctor'], bins=60)
plt.xlabel('Click-to-Open Rates')
plt.ylabel('Subscriber Counts (Log Scale)')
plt.yscale('log')
plt.title('Distribution of Click-to-Open Rates')

The `sent` distribution shows how many campaigns each subscriber has received. This will be used to inform the minimum send threshold applied during sampling. Subscribers with too few sends don't have enough history to produce reliable `ctor` signals.

In [ ]:
# Plotting the histogram for sent
plt.hist(emails_sent_df['sent'], bins=100)
plt.xlabel('Send Counts')
plt.ylabel('Subscriber Counts (Log Scale)')
plt.yscale('log')
plt.xticks(range(0, 260, 20))
plt.title('Distribution of Email Counts')

Random sampling from the full list would undersample bot-behavior contacts since more than half have a `ctor` of 0. The three groups intentionally oversample the high-suspicion group to give the classifier sufficient bot-behavior examples to learn from.

All tiers require at least one send — subscribers who have never received a campaign are excluded since there is no engagement history to measure.

The cutoff between mid and high suspicion is set at 65% based on the CTOR distribution, which shows a distinct second concentration above that threshold.

In [ ]:
# Stratification
# Using pd.sample to select samples at equal chance rather than random chance with random.sample
low_sus = emails_sent_df[emails_sent_df['ctor'] == 0].sample(300)

# Cannot use df>0<65, py cannot evaluate a Pandas Series or DataFrame containing multiple elements into a single T/F value
# An ambiguous error will happen
# Building masks separately to make the filtering logic explicit
# Each mask is a boolean Series - one True/False value per row
mask1 = emails_sent_df['ctor'] > 0
mask2 = emails_sent_df['ctor'] < 65
mid_sus = emails_sent_df[mask1 & mask2].sample(300)

high_sus = emails_sent_df[emails_sent_df['ctor'] >= 65].sample(400)

print(len(low_sus))
print(len(mid_sus))
print(len(high_sus))

In [ ]:
# Checking the min and mx of ctor in each df
display(low_sus.describe())
display(mid_sus.describe())
display(high_sus.describe())

In [ ]:
# Concatenating the three dfs into one and save it as a csv for activity log fetch
sample_df = pd.concat([low_sus, mid_sus, high_sus], ignore_index=True)
print(sample_df.shape)
sample_df.head()

In [ ]:
sample_df.to_csv('1k_sub_sample.csv', index=False)